# Lecture 24: Interpreting Confidence
v.ekc

Estimation, continued: we have *one* sample and want a population parameter. The bootstrap gives an estimate **and** its give-or-take; then the fine print: what "95% confident" actually means, what determines the width of an interval, care in using the method, and how an interval doubles as a hypothesis test.

**Sections (two class days):**

*Monday 10/26*
1. The bootstrap — SF compensation
2. A confidence interval for the population mean

*Wednesday 10/28*

3. What "95% confident" means and what sets the width: sample size, confidence level


In [ ]:
from datascience import *
%matplotlib inline
path_data = '../../../assets/data/'
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import numpy as np
import warnings
warnings.filterwarnings("ignore")

---
## 1. The Bootstrap 👢: $F Compensation

Friday we met percentiles and a population we can see: San Francisco city workers' 2019 pay. One random sample gives one estimate of the population median, but a different sample would give a different number. **How far off might our one estimate be?** Quantifying that seems to require many samples, and in the real world we can't go back to the population.

The bootstrap's idea: **the sample resembles the population, so resample from the sample.**

### 📋 Reference Table

| Bootstrap rule | Why |
|---|---|
| draw from the **original sample** | it's our best stand-in for the population |
| **with replacement** | otherwise you'd just get the same sample back |
| **same size** as the original | variability depends on sample size |
| `our_sample.sample()` | does all three by default! |
| `percentile(2.5, stats)`, `percentile(97.5, stats)` | the middle 95% of the bootstrap estimates = a 95% confidence interval |

In [ ]:
sf = Table.read_table('san_francisco_2019.csv')
min_salary = 15 * 20 * 50
sf = sf.where('Salary', are.above(min_salary))

In [ ]:
sf_bins = np.arange(0, 726000, 25000)
sf.hist('Total Compensation', bins=sf_bins)

The parameter, wrapped in a small function this time (we will reuse it):

In [ ]:
# Parameter: Median total compensation in the population
def median_comp(t):
    return percentile(50, t.column('Total Compensation'))

median_comp(sf)

Now pretend all we get is one sample of 400:

In [ ]:
our_sample = sf.sample(400, with_replacement=False)

In [ ]:
our_sample.hist('Total Compensation', bins=sf_bins)

The sample median, our one-number estimate:

In [ ]:
pop_median = percentile(50, sf.column('Total Compensation'))

sample_median = percentile(50, our_sample.column('Total Compensation') )

print(f'''Full population median: {pop_median}
Our sample of 400: {sample_median}''')

### Bootstrap

Sample randomly
 - from the original sample
 - with replacement
 - the same number of times as the original sample size

In [ ]:
# Default behavior of tbl.sample:
# at random with replacement,
# the same number of times as rows of tbl

bootstrap_sample = our_sample.sample()

In [ ]:
bootstrap_sample.hist('Total Compensation', bins=sf_bins)

### Bootstrap Sample Median
This is one estimate of the population median.

In [ ]:
percentile(50, bootstrap_sample.column('Total Compensation'))

Package the move into a function:

In [ ]:
def one_bootstrap_median():
    # draw the bootstrap sample
    resample = our_sample.sample()
    # return the median total compensation in the bootstrap sample
    return percentile(50, resample.column('Total Compensation'))

In [ ]:
one_bootstrap_median()

A thousand bootstrap medians:

In [ ]:
# Generate the medians of 1000 bootstrap samples
num_repetitions = 1000
bstrap_medians = make_array()
for i in np.arange(num_repetitions):
    bstrap_medians = np.append(bstrap_medians, one_bootstrap_median())

Histogram them; the red dot marks the parameter we secretly know:

In [ ]:
resampled_medians = Table().with_column('Bootstrap Sample Median', bstrap_medians)
median_bins=np.arange(120000, 160000, 2000)
resampled_medians.hist(bins = median_bins)

# our_sample median statistic
our_sample_med = percentile(50, bootstrap_sample.column('Total Compensation'))

# Plotting parameters; you can ignore this code
parameter_green = '#32CD32'
plots.ylim(-0.000005, 0.00014)
plots.scatter(pop_median, 0, color='green', s=40, zorder=2)
plots.scatter(our_sample_med, 0, color='red', s=40, zorder=2)
plots.title('Bootstrap Medians and the Parameter (Green Dot)');

### Percentile Method: Middle 95% of the Bootstrap Estimates

In [ ]:
left = percentile(2.5, bstrap_medians)
right = percentile(97.5, bstrap_medians)

make_array(left, right)

In [ ]:
resampled_medians.hist(bins = median_bins)

# Plotting parameters; you can ignore this code
plots.ylim(-0.000005, 0.00014)
plots.plot(make_array(left, right), make_array(0, 0), color='yellow', lw=3, zorder=1)
plots.scatter(pop_median, 0, color='green', s=40, zorder=2)
plots.scatter(our_sample_med, 0, color='red', s=40, zorder=2)
plots.title('Bootstrap Medians and the Parameter (Green Dot) and 95% CI');

### Try It 1: Shrink the sample

1. Draw one sample of just **100** employees (without replacement) and call it `small_sample`.

2. Bootstrap 1,000 medians from `small_sample` and take the middle 95%. (Your helper function must resample `small_sample`, not `our_sample`.)

3. Compare your interval to the 400-person interval above. Which is wider, and why does that make sense?

In [ ]:
# 1. sample of 100


In [ ]:
# 2. bootstrap the small sample, middle 95%


*3. Your interpretation: (double-click to edit)*

<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Same three bootstrap rules; only the original sample changed.*

```python
# 1.
small_sample = sf.sample(100, with_replacement=False)

# 2.
def one_small_median():
    return percentile(50, small_sample.sample().column('Total Compensation'))

small_medians = make_array()
for i in np.arange(1000):
    small_medians = np.append(small_medians, one_small_median())

make_array(percentile(2.5, small_medians), percentile(97.5, small_medians))
```

*3. The 100-person interval is usually much wider: smaller samples vary more, and the bootstrap faithfully reports that extra uncertainty.*

</details>

> **Looking ahead to Project 2:** a confidence interval doubles as a two-sided hypothesis test. To test "the parameter equals $x$" at the 5% cutoff, build the 95% interval: if $x$ is **not** in it, reject the null; if it is, fail to reject. At the 1% cutoff, use the 99% interval: `percentile(0.5, stats)` to `percentile(99.5, stats)`. You will see this in Project 2, Part 1 

---
# ⏩ Wednesday 10/28 starts here??

---
## 3. What "95% Confident" Means 🎯

We just said "we are 95% confident that the population mean age is between 26.9 and 27.6." Confident in *what*, exactly? The population mean is one fixed number; it is either in that interval or it isn't.

The textbook's answer (Ch. 13.3): we are 95% confident **that the process results in a good interval**. To see what that means we need a population we can check against, so back to SF compensation, where we know the median and can replicate the whole process of estimation many times.

First, package the whole bootstrap into one reusable function. Nothing new here, it is Friday's template with the statistic and the confidence level as inputs.

### 📋 Reference Table

| Code | What it does |
|---|---|
| `bootstrap_stats(sample, 'col', statistic)` | 1,000 bootstrap values of `statistic` |
| `bootstrap_ci(sample, 'col', statistic, level)` | the middle `level`% of those values |
| `level = 95` | cut 2.5% off each end → `percentile(2.5, …)`, `percentile(97.5, …)` |
| `level = 80` | cut 10% off each end → `percentile(10, …)`, `percentile(90, …)` |
| **"95% confident"** | the process of estimation captures the parameter about 95 out of every 100 times we use it |

In [ ]:
def sample_median(arr):
    return percentile(50, arr)

def bootstrap_stats(original_sample, label, statistic, reps=1000):
    """Resample the original sample `reps` times; return the statistic from each."""
    stats = make_array()
    for i in np.arange(reps):
        resample = original_sample.sample()
        stats = np.append(stats, statistic(resample.column(label)))
    return stats

def bootstrap_ci(original_sample, label, statistic, level=95, reps=1000):
    """The middle `level`% of the bootstrap statistics."""
    stats = bootstrap_stats(original_sample, label, statistic, reps)
    tail = (100 - level) / 2
    return make_array(percentile(tail, stats), percentile(100 - tail, stats))

Check: the same interval we built by hand in §1, in one line.

In [ ]:
bootstrap_ci(our_sample, 'Total Compensation', sample_median, 95)

### A Hundred Intervals

The experiment: draw a **fresh** random sample of 100 employees from the population, bootstrap it, keep the 95% interval. Repeat 100 times. How many of the 100 intervals catch the true median?

(Each interval uses 500 resamples to keep this under a minute. The helper below just loops; the plotting function after it you can ignore.)

In [ ]:
def many_intervals(sample_size, statistic, level=95, how_many=100):
    """`how_many` fresh samples from sf, each bootstrapped into a `level`% CI."""
    intervals = Table(['Number', 'lower', 'upper'])
    for k in np.arange(how_many):
        fresh_sample = sf.sample(sample_size, with_replacement=False)
        ci = bootstrap_ci(fresh_sample, 'Total Compensation', statistic, level, reps=500)
        intervals = intervals.with_row([k + 1, ci.item(0), ci.item(1)])
    return intervals

def hit_rate(intervals, truth):
    """Proportion of the intervals that contain `truth`."""
    caught = intervals.where('lower', are.below_or_equal_to(truth)).where('upper', are.above_or_equal_to(truth))
    return caught.num_rows / intervals.num_rows

In [ ]:
# Plotting code; you can ignore this
def plot_intervals(intervals, truth, group_label=None, xlabel='Total Compensation'):
    plots.figure(figsize=(8, 6))
    palette = ['#1f77b4', '#ff7f0e', '#9467bd', '#8c564b']   # no green: green is the parameter
    groups = [] if group_label is None else sorted(set(intervals.column(group_label)))
    seen = set()
    for row in intervals.rows:
        lo, hi, n = row.item('lower'), row.item('upper'), row.item('Number')
        if lo > truth or hi < truth:
            color, label = 'red', None                      # missed the parameter
        elif group_label is None:
            color, label = 'black', None
        else:
            g = row.item(group_label)
            color = palette[groups.index(g) % len(palette)]
            label = None if g in seen else f'{group_label} = {g}'
            seen.add(g)
        plots.plot((lo, hi), (n, n), color=color, lw=1.8, label=label)
    plots.plot((truth, truth), (0, intervals.num_rows + 1), color=parameter_green, lw=3)
    plots.yticks([])
    plots.xlabel(xlabel)
    plots.ylabel('one interval per line')
    if group_label is not None:
        plots.legend(loc='upper left', bbox_to_anchor=(1.01, 1))
    misses = intervals.num_rows - round(hit_rate(intervals, truth) * intervals.num_rows)
    plots.title(f'{intervals.num_rows} intervals; {misses} miss the parameter (red)');

In [ ]:
intervals_100 = many_intervals(100, sample_median)
intervals_100.show(5)

In [ ]:
plot_intervals(intervals_100, pop_median)

In [ ]:
hit_rate(intervals_100, pop_median)

> **What "95% confident" means (Ch. 13.3):** our process of estimation (random sample → bootstrap → middle 95%) captured the parameter about 95 out of every 100 times we used it. That is what we are 95% confident in: **the process that generated the interval**, not any one interval. In real life we get **one** sample and **one** interval, and we never find out whether ours is a black line or a red one.
>
> **A common misuse (Ch. 13.4):** concluding that about 95% of the mothers were between 26.9 and 27.6 years old. The interval just estimates **one number**, the average of all the ages in the population; it says nothing about where individual mothers fall (scroll up to the yellow strip on the age histogram in §2, it covers almost nobody). Likewise, it is not a 95% chance that the parameter is in *this* interval: the parameter is fixed, the interval is what varies.

### Try It 3: Reading the picture 😊

1. Re-run the `many_intervals` cell and the plot. Did the number of red intervals change? Did the green line move? Explain the difference.

2. True or false: "About 95% of the 37,103 employees have a Total Compensation inside our interval from `our_sample`." Write one sentence saying why.

3. True or false: "There is a 95% chance that the population median is inside the interval we got from `our_sample`." Write one sentence saying why.

*1. Your observation: (double-click to edit)*

*2. True / False, because: (double-click to edit)*

*3. True / False, because: (double-click to edit)*

<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*1. The red count changes a little (it is a random experiment; 3, 5, 7 misses are all normal) and the green line never moves: the parameter is fixed, the intervals are random.*

*2. False. The interval is about the **median**, a single number describing the population, not about where individual employees fall. Roughly half of employees are above the median and half below, so most are outside a narrow interval around it.*

*3. False, at least not the way "chance" is used in this class. Once the sample is drawn, the interval is a fixed pair of numbers and the parameter is a fixed number, so the parameter is either in it (100%) or not (0%). The 95% describes how often the **method** succeeds across many samples.*

</details>

### Sample Size and Width 📏

Try It 1 compared one 100-person interval to one 400-person interval. Now 20 intervals at each of three sample sizes, same confidence level (95%).

**Predict first:** which group of intervals will be widest? Will the proportion that capture the parameter change?

In [ ]:
by_size = Table(['Number', 'lower', 'upper', 'sample size'])

for n in [25, 100, 400]:
    batch = many_intervals(n, sample_median, level=95, how_many=20)
    batch = batch.with_column('sample size', n)
    by_size = by_size.with_rows(batch.rows)

by_size = by_size.with_column('Number', np.arange(1, by_size.num_rows + 1))
plot_intervals(by_size, pop_median, group_label='sample size')

In [ ]:
# average width of the 95% interval at each sample size
by_size.with_column('width', by_size.column('upper') - by_size.column('lower')).group('sample size', np.average).select('sample size', 'width average')

> **Sample size:** a larger original sample gives a *narrower* interval at the same 95% confidence level. The sample averages (and medians) of large samples don't vary much, so the bootstrap distribution is tighter; the width shrinks roughly like $1/\sqrt{n}$, so four times the sample size is about half the width. The confidence level did not change, so the proportion of intervals that capture the parameter did not either (about 95 out of 100 at every size).

### Confidence Level and Width

Now hold the sample size fixed at 100 and change what we trim, 40 intervals at each level. An 80% interval keeps the middle 80% of the bootstrap medians (cut 10% off each end); a 99% interval keeps the middle 99% (cut 0.5% off each end).

**Predict first:** which group is widest? Which group misses the parameter most often?

In [ ]:
by_level = Table(['Number', 'lower', 'upper', 'level'])

for level in [80, 90, 95, 99]:
    batch = many_intervals(100, sample_median, level=level, how_many=40)
    batch = batch.with_column('level', level)
    by_level = by_level.with_rows(batch.rows)

by_level = by_level.with_column('Number', np.arange(1, by_level.num_rows + 1))
plot_intervals(by_level, pop_median, group_label='level')

In [ ]:
# how often each level captures the parameter: 40 intervals each, so 1 miss = 2.5%
for level in [80, 90, 95, 99]:
    group = by_level.where('level', level)
    print(f'{level}% intervals: captured the median {round(hit_rate(group, pop_median) * 100)}% of the time')

> **Confidence level (Ch. 13.3):** the 80% confidence interval is much shorter than the 95% confidence interval, but we have less confidence in the process that generated it: about 1 in 5 of the 80% intervals miss the parameter, versus about 1 in 20 of the 95% intervals and almost none of the 99% intervals. A higher confidence level buys a wider interval. **An 80% interval is shorter than a 95% one, not wider**, because you trimmed more off each end. Same sample, same bootstrap values, just different percentiles. On the mothers' ages from §2, from the same `bstrap_means`: 80% ≈ (27.0, 27.4) · 95% ≈ (26.9, 27.6) · 99% ≈ (26.8, 27.7).
>
> Putting the two together, in the textbook's words: *to get a narrow confidence interval at a high level of confidence, you'll have to start with a larger sample.*

---
## 2. A Confidence Interval for the Population Mean

Same recipe, new statistic (the mean), new data: 1,174 mother-newborn pairs.

In [ ]:
# Random sample of mother-newborn pairs
births = Table.read_table('baby.csv')
births

In [ ]:
births.hist('Maternal Age')

The sample average, the estimate we will build an interval around:

In [ ]:
# Average age of mothers in the sample
np.average(births.column('Maternal Age'))

### Question
What is the average age of the mothers in the population?

In [ ]:
def one_bootstrap_mean():
    resample = births.sample()
    return np.average(resample.column('Maternal Age'))

In [ ]:
# Generate means from 3000 bootstrap samples
num_repetitions = 3000
bstrap_means = make_array()
for i in np.arange(num_repetitions):
    bstrap_means = np.append(bstrap_means, one_bootstrap_mean())

### Bootstrap Percentile Method for Confidence Interval

The interval of estimates is the "middle 95%" of the bootstrap estimates.

This is called a *95% confidence interval* for the mean age in the population.

In [ ]:
# Get the endpoints of the 95% confidence interval
left = percentile(2.5, bstrap_means)
right = percentile(97.5, bstrap_means)

make_array(left, right)

In [ ]:
resampled_means = Table().with_columns(
    'Bootstrap Sample Mean', bstrap_means
)
resampled_means.hist(bins=15)
plots.plot([left, right], [0, 0], color='yellow', lw=8);

Now the same yellow interval drawn on the data's own histogram (this picture matters for the fine print below):

In [ ]:
births.hist('Maternal Age')
plots.plot([left, right], [0, 0], color='yellow', lw=8);

### Try It 2: Baby weights 😊

1. Bootstrap a 95% confidence interval for the **average Birth Weight** in the population (3,000 resamples).

2. Finish this sentence precisely: "We are 95% confident that ______." What exactly is the 95% about?

In [ ]:
# 1. bootstrap CI for average birth weight


*2. Your interpretation: (double-click to edit)*

<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Swap the column, keep the skeleton.*

```python
# 1.
def one_bootstrap_weight():
    resample = births.sample()
    return np.average(resample.column('Birth Weight'))

bstrap_weights = make_array()
for i in np.arange(3000):
    bstrap_weights = np.append(bstrap_weights, one_bootstrap_weight())

make_array(percentile(2.5, bstrap_weights), percentile(97.5, bstrap_weights))
```

*2. "...the average birth weight in the population is between about 118.4 and 120.5 ounces." The 95% describes the method, not this one interval: intervals built this way catch the true average for about 95% of samples.*

</details>

---
# 📖 Reading: §4 and §5 go with Homework 7
*Run the two cases in §4 yourself (about 15 seconds); every question from here on has a 💡 toggle.*

---
## 4. Care in Using the Bootstrap Percentile Method 🧱

The bootstrap percentile method works well for estimating the population median or mean based on a **large random sample**. It works because the sample resembles the population, so resampling the sample resembles resampling the population. When that resemblance fails, so does the interval. Since we still have the SF population, we can watch it fail.

### 📋 Reference Table

The textbook (Ch. 13.3) says the method is **not expected to do well** when:

| Situation | Why | The clue |
|---|---|---|
| the goal is to estimate the **minimum or maximum** value in the population (or another extreme percentile) | a sample almost never contains the population's extreme, and resampling can't produce values that aren't in the sample | the bootstrap histogram is lopsided, not bell-shaped |
| the **original sample is very small**, say less than 10 or 15 | a few values can't stand in for the population; the interval is narrow *and* often wrong | far fewer than 95 out of 100 intervals capture the parameter |
| the **probability distribution of the statistic is not roughly bell-shaped** | the percentile method assumes a reasonably symmetric distribution of estimates | look at the bootstrap histogram before you trust the endpoints |

Start with a large random sample. If you don't, the method might not work.

### Case 1: Estimating the Maximum

Same `our_sample` of 400 as §1, but now the parameter is the **highest** Total Compensation in the city.

In [ ]:
pop_max = max(sf.column('Total Compensation'))
sample_max = max(our_sample.column('Total Compensation'))

print(f"""Population maximum: {pop_max}
Our sample's maximum: {sample_max}""")

In [ ]:
bstrap_maxes = bootstrap_stats(our_sample, 'Total Compensation', max)

Table().with_column('Bootstrap Sample Max', bstrap_maxes).hist(bins=20)
plots.scatter(pop_max, 0.000005, color=parameter_green, s=40, zorder=2)
plots.title('Bootstrap Maxes and the Parameter (Green Dot)');

In [ ]:
bootstrap_ci(our_sample, 'Total Compensation', max, 95)

> The green dot is off to the right, usually not even inside the interval. A bootstrap sample can never produce a value **larger than the sample's own maximum**, so every one of the 1,000 estimates is at or below `sample_max`. 400 people out of 37,103 almost never include the single top earner. Notice the histogram too: a wall on the right, a tail on the left, nothing like a bell.

### Case 2: A Very Small Original Sample

Now the mean, which the bootstrap usually handles well, but from samples of only **5** employees. Each small sample gets bootstrapped into a 95% interval; we build 100 of them and count how many capture the parameter, exactly like §3.

In [ ]:
pop_mean = np.average(sf.column('Total Compensation'))
pop_mean

In [ ]:
tiny_intervals = many_intervals(5, np.average, level=95)
plot_intervals(tiny_intervals, pop_mean)

In [ ]:
hit_rate(tiny_intervals, pop_mean)

> A 95% confidence level, but only about 75–85 of the 100 intervals capture the parameter. Five values can't tell you the shape of 37,000, so the bootstrap intervals come out too narrow and miss far more often than 5 out of 100. **The bootstrap measures the variability *in your sample*; it can't know about the variability your sample never saw.** That is why the textbook says to start with a large random sample: a few hundred is comfortable, a few dozen is iffy, fewer than 10 or 15 is not a bootstrap problem.

### Discussion: Would you trust it? 🤔

For each, say *yes*, *no*, or *check the histogram first*, and name the row of the reference table.

1. A 95% CI for the **median** rent in Arcata from a random sample of 300 listings.
2. A 95% CI for the **minimum** commute time in Humboldt County from a random sample of 500 workers.
3. A 95% CI for the **mean** final-exam score from the 8 students who showed up to office hours.
4. A 95% CI for the **90th percentile** of Total Compensation from `our_sample`.

<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*1. Yes: a middle-of-the-distribution statistic, a decent sample size.*

*2. No: the minimum depends on the rarest person, the same problem as the max.*

*3. No: n = 8 is Case 2 (less than 10 or 15), and office-hours attendees aren't a random sample anyway (a different problem the bootstrap can't fix either).*

*4. Check the histogram first: the 90th percentile is fairly extreme but 400 people do reach into that region; if the bootstrap histogram looks roughly bell-shaped, go ahead, with caution.*

</details>

---
## 5. Using the Confidence Interval for Testing Hypotheses

A confidence interval has a single purpose, to estimate an unknown parameter, but there is a *duality* between confidence intervals and tests (Ch. 13.4): the interval can also be used to test a hypothesis about the parameter. Back to the mothers in §2.

**Null:** The average age of mothers in the population is 25 years; the random sample average is different due to chance.

**Alternative:** The average age of the mothers in the population is not 25 years.

Suppose you use the 5% cutoff for the p-value.

Based on the 95% confidence interval, which hypothesis would you pick?

In [ ]:
# the 95% CI for mean maternal age, from §2
make_array(percentile(2.5, bstrap_means), percentile(97.5, bstrap_means))

<details>
<summary>💡 Answer — click to peek</summary>
<br>

*25 is well outside (26.9, 27.6), so the data are inconsistent with the null: **reject the null at the 5% level.** Values the data are consistent with are exactly the values inside the interval.*

*The rule (Ch. 13.4): if you are testing whether the population mean is a particular value $x$ and you use the 5% cutoff for the p-value, you reject the null hypothesis if $x$ is not in your 95% confidence interval for the mean. In practice it boils down to checking whether the value specified in the null hypothesis lies in the interval. Generalized: to test at the p% cutoff, build the (100 − p)% interval. A 1% cutoff uses the 99% interval, `percentile(0.5, …)` to `percentile(99.5, …)`, which here is about (26.8, 27.7), so 25 is rejected at 1% too.*

</details>

### Try It 4: CI as a hypothesis test 😊

**Null:** 40% of mothers in the population smoked during pregnancy. **Alternative:** it wasn't 40%. Use the **5% cutoff**.

1. Bootstrap a 95% confidence interval for the proportion of smokers (3,000 resamples). Is 0.40 inside? What do you conclude?

In [ ]:
# 1. bootstrap CI for the smoking proportion


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Same skeleton as the mean, only the statistic changes. The interval comes out around (0.36, 0.42), which contains 0.40 → the data are consistent with the null; fail to reject at the 5% level.*

```python
# 1.
def one_bootstrap_prop():
    resample = births.sample()
    return np.count_nonzero(resample.column('Maternal Smoker')) / resample.num_rows

props = make_array()
for i in np.arange(3000):
    props = np.append(props, one_bootstrap_prop())

make_array(percentile(2.5, props), percentile(97.5, props))
```

</details>

---
> **Today's takeaway:** "95% confident" is confidence in the **process**: it captures the parameter about 95 out of every 100 times we use it. It is not a statement about the data (not "95% of mothers") or about this one interval. Larger samples make the interval narrower; a higher confidence level makes it wider; very small samples and extreme statistics like the max are where the bootstrap percentile method is not expected to do well. And by the duality with tests: if the null value is not in the 95% interval, reject at the 5% cutoff. Homework 7's interpretation questions come straight from today's fine print.

## Appendix — Quick Reference

Full `datascience` docs: [Textbook, Chapter 13.3–13.4: Confidence Intervals](https://inferentialthinking.com/chapters/13/3/Confidence_Intervals.html)

### General bootstrap CI
```python
def bootstrap_ci(original_sample, label, statistic, level=95, reps=1000):
    stats = make_array()
    for i in np.arange(reps):
        stats = np.append(stats, statistic(original_sample.sample().column(label)))
    tail = (100 - level) / 2
    return make_array(percentile(tail, stats), percentile(100 - tail, stats))
```

### What sets the width
| Factor | Increase it… | Interval gets… | Captures the parameter… |
|---|---|---|---|
| sample size $n$ | 100 → 400 | narrower (≈ half, like $1/\sqrt{n}$) | just as often (same level) |
| confidence level | 80% → 99% | wider | more often |

### Care in using the bootstrap percentile method
start with a large random sample · not for the min / max (or other extreme percentiles) · not if the bootstrap histogram isn't roughly bell-shaped · not if the original sample is very small (less than 10 or 15)

### CI ↔ hypothesis test
```python
# reject "parameter = v" at the 5% level  <=>  v outside the 95% CI
# reject at the 1% level                  <=>  v outside the 99% CI
```